In [1]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

Found existing installation: keras 3.10.0
Uninstalling keras-3.10.0:
  Successfully uninstalled keras-3.10.0
Found existing installation: matplotlib 3.10.0
Uninstalling matplotlib-3.10.0:
  Successfully uninstalled matplotlib-3.10.0
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.simplefilter('ignore')

In [3]:
import os
import sys
import subprocess

In [4]:
import glob

def set_env(input_archive, temp_dir):

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    whl_files = sorted(glob.glob(f'{temp_dir}/wheels/*.whl'))
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-deps', 
        *whl_files
    ], check=True)

In [5]:
set_env(
    input_archive='/kaggle/input/notebooks/nahidhossainredom/aimo-utils-qwen-3-5/wheels.tar.gz', 
    temp_dir='/kaggle/tmp/setup'
)

Processing /kaggle/tmp/setup/wheels/aiohappyeyeballs-2.6.1-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/aiohttp-3.13.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
Processing /kaggle/tmp/setup/wheels/aiosignal-1.4.0-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/annotated_doc-0.0.4-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/annotated_types-0.7.0-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/anthropic-0.84.0-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/anyio-4.12.1-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/apache_tvm_ffi-0.1.9rc2-cp312-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl
Processing /kaggle/tmp/setup/wheels/astor-0.8.1-py2.py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/attrs-25.4.0-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/blake3-1.0.8-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/tmp/setup/wheels/cachetools-7.0.1-py3-none-any.whl
Process

In [6]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'

In [7]:
import gc
import re
import json
import math
import time
import queue
import threading
import contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

In [8]:
class CFG:
    
    system_prompt = (
        'You are an elite mathematical problem solver with expertise at the International '
        'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
        'rigorous mathematical reasoning.\n\n'
        
        '# Problem-Solving Approach:\n'
        '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
        'Identify what is given, what needs to be found, and any constraints.\n'
        '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
        'techniques, patterns, or analogous problems. Don\'t commit to one approach immediately.\n'
        '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
        '4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.\n'
        '5. VERIFY: Check your answer by substituting back, testing edge cases, or using '
        'alternative methods. Ensure logical consistency throughout.\n\n'
        
        '# Mathematical Reasoning Principles:\n'
        '- Break complex problems into smaller, manageable sub-problems\n'
        '- Look for patterns, symmetries, and special cases that provide insight\n'
        '- Use concrete examples to build intuition before generalizing\n'
        '- Consider extreme cases and boundary conditions\n'
        '- If stuck, try working backwards from the desired result\n'
        '- Be willing to restart with a different approach if needed\n\n'
        
        '# Verification Requirements:\n'
        '- Cross-check arithmetic and algebraic manipulations\n'
        '- Verify that your solution satisfies all problem constraints\n'
        '- Test your answer with simple cases or special values when possible\n'
        '- Ensure dimensional consistency and reasonableness of the result\n\n'
        
        '# Output Format:\n'
        'The final answer must be a non-negative integer between 0 and 99999.\n'
        'Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}\n\n'
        
        'Think step-by-step and show your complete reasoning process. Quality of reasoning '
        'is as important as the final answer.'
    )
    
    tool_description = (
        'Execute Python code in a stateful Jupyter notebook environment.\n'
        'Use this for:\n'
        '- Complex calculations that would be error-prone by hand\n'
        '- Numerical verification of analytical results\n'
        '- Generating examples or testing conjectures\n'
        '- Brute-force verification for small cases\n\n'
        'Available libraries: math, numpy, sympy, itertools, collections, mpmath.\n'
        'Always use print() to display results. Code persists between executions.'
    )
    
    preference_prompt = (
        'You have access to a Python tool for computation. Use it when helpful.\n'
        'Available libraries: `math`, `numpy`, `sympy`, `itertools`, `collections`, `mpmath`.\n\n'
        
        '# Symbolic Computation (sympy):\n'
        '- Algebraic manipulation and simplification\n'
        '- Solving equations and systems of equations\n'
        '- Number theory functions (primes, divisors, modular arithmetic)\n'
        '- Polynomial operations and factorization\n\n'
        
        '# Numerical Computation (numpy):\n'
        '- Array operations and linear algebra\n'
        '- Efficient numerical calculations for large datasets\n\n'
        
        'Best Practices:\n'
        '- Use sympy for exact symbolic answers when possible\n'
        '- Use numpy for numerical verification and large-scale computation\n'
        '- Combine symbolic and numerical approaches: derive symbolically, verify numerically\n'
        '- Validate computational results against known cases or theoretical bounds'
    )
    
    served_model_name = 'qwen3.5-27b'
    model_path = '/kaggle/input/models/qwen-lm/qwen-3-5/transformers/qwen3.5-27b/1'
    
    dtype = 'auto'

    high_problem_timeout = 900
    base_problem_timeout = 300

    notebook_limit = 17600
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 6
    sandbox_timeout = 3

    context_tokens = 65536
    buffer_tokens = 512
    search_tokens = 32
    top_logprobs = 5
    batch_size = 64
    early_stop = 4
    attempts = 8
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.95
    temperature = 1.0
    top_p = 0.95
    top_k = 20
    presence_penalty = 1.5

In [9]:
set_seed(CFG.seed)

In [10]:
PYTHON_TOOL_DEFINITION = {
    'type': 'function',
    'function': {
        'name': 'python',
        'description': CFG.tool_description,
        'parameters': {
            'type': 'object',
            'properties': {
                'code': {
                    'type': 'string',
                    'description': 'Python code to execute'
                }
            },
            'required': ['code']
        }
    }
}

In [11]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):
        
        self.execute(
            '%reset -f\n'
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def __del__(self):

        self.close()

In [12]:
class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, sandbox=None):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._jupyter_session = sandbox
        
        self._owns_session = sandbox is None
        
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:

        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    def execute(self, code: str) -> str:

        self._ensure_session()
        final_script = self._ensure_last_print(code)

        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)

            except TimeoutError as exc:
                output = f'[ERROR] {exc}'

        return output

In [13]:
class AIMO3Solver:

    def __init__(self, cfg, port: int = 8000):
    
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
    
        self._preload_model_weights()
        
        self.server_process = self._start_server()
    
        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )
    
        self._wait_for_server()
        self._initialize_kernels()
    
        self.notebook_start_time = time.time()
        self.problems_remaining = 50
    
    def _preload_model_weights(self) -> None:
    
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        
        files_to_load = []
        total_size = 0
    
        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
    
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)
    
        def _read_file(path: str) -> None:
    
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))
    
        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')
    
    def _start_server(self) -> subprocess.Popen:
    
        cmd = [
            sys.executable, 
            '-m', 
            'vllm.entrypoints.openai.api_server', 
            '--seed', 
            str(self.cfg.seed), 
            '--model', 
            self.cfg.model_path, 
            '--served-model-name', 
            self.cfg.served_model_name, 
            '--tensor-parallel-size', 
            '1', 
            '--max-num-seqs', 
            str(self.cfg.batch_size), 
            '--gpu-memory-utilization', 
            str(self.cfg.gpu_memory_utilization), 
            '--host', 
            '0.0.0.0', 
            '--port', 
            str(self.port), 
            '--dtype', 
            self.cfg.dtype, 
            '--max-model-len', 
            str(self.cfg.context_tokens), 
            '--reasoning-parser', 
            'qwen3', 
            '--enable-auto-tool-choice', 
            '--tool-call-parser', 
            'qwen3_coder', 
            '--language-model-only', 
            '--enable-prefix-caching', 
            '--disable-log-stats'
        ]
    
        self.log_file = open('vllm_server.log', 'w')
    
        return subprocess.Popen(
            cmd, 
            stdout=self.log_file, 
            stderr=subprocess.STDOUT, 
            start_new_session=True
        )
    
    def _wait_for_server(self):
    
        print('Waiting for vLLM server...')
        start_time = time.time()
    
        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()
    
            if return_code is not None:
                self.log_file.flush()
    
                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()
    
                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')
    
            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')
    
                return
    
            except Exception:
                time.sleep(1)
    
        raise RuntimeError('Server failed to start (timeout).\n')
    
    def _initialize_kernels(self) -> None:
    
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()
    
        self.sandbox_pool = queue.Queue()
    
        def _create_sandbox():
            
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]
    
            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())
    
        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')
    
    def _scan_for_answer(self, text: str) -> int | None:
        
        pattern = r'\\boxed\s*\{\s*([0-9,]+)\s*\}'
        matches = re.findall(pattern, text)
    
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
    
                if 0 <= value <= 99999:
                    return value
    
            except ValueError:
                pass
                
        pattern = r'final\s+answer\s+is\s*([0-9,]+)'
        matches = re.findall(pattern, text, re.IGNORECASE)
    
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
    
                if 0 <= value <= 99999:
                    return value
    
            except ValueError:
                pass
    
        return None
    
    def _compute_mean_entropy(self, logprobs_list: list) -> float:
    
        if not logprobs_list:
            return float('inf')
    
        total_entropy = 0.0
        token_count = 0
    
        for token_logprob_info in logprobs_list:
            
            if not hasattr(token_logprob_info, 'top_logprobs') or not token_logprob_info.top_logprobs:
                continue
            
            token_entropy = 0.0
            
            for top_lp in token_logprob_info.top_logprobs:
                prob = math.exp(top_lp.logprob)
                
                if prob > 0:
                    token_entropy -= prob * math.log2(prob)
            
            total_entropy += token_entropy
            token_count += 1
    
        if token_count == 0:
            return float('inf')
    
        return total_entropy / token_count

    def _extract_tool_code(self, tool_calls) -> str | None:

        if not tool_calls:
            return None
    
        for tc in tool_calls:
            if tc.function.name == 'python':
                try:
                    args = json.loads(tc.function.arguments)
                    return args.get('code', '')
                except (json.JSONDecodeError, AttributeError):
                    return tc.function.arguments
    
        return None

    def _process_attempt(
        self, 
        problem: str, 
        system_prompt: str, 
        attempt_index: int, 
        stop_event: threading.Event, 
        deadline: float
    ) -> dict:
    
        if stop_event.is_set() or time.time() > deadline:
            return {
                'Attempt': attempt_index + 1, 
                'Answer': None, 
                'Python Calls': 0, 
                'Python Errors': 0, 
                'Response Length': 0, 
                'Entropy': float('inf')
            }
    
        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None
        
        logprobs_buffer = []
    
        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))
    
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
    
            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout, 
                sandbox=sandbox
            )
    
            messages = [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': problem}
            ]
    
            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    break
    
                try:
                    response = self.client.chat.completions.create(
                        model=self.cfg.served_model_name, 
                        messages=messages,
                        tools=[PYTHON_TOOL_DEFINITION],
                        temperature=self.cfg.temperature, 
                        top_p=self.cfg.top_p,
                        max_completion_tokens=self.cfg.context_tokens,
                        seed=attempt_seed,
                        logprobs=True,
                        top_logprobs=self.cfg.top_logprobs,
                        extra_body={
                            'top_k': self.cfg.top_k, 
                            'presence_penalty': self.cfg.presence_penalty
                        }
                    )
                except Exception:
                    break

                choice = response.choices[0]
                assistant_message = choice.message
                
                if response.usage:
                    total_tokens += response.usage.completion_tokens or 0
                
                if choice.logprobs and choice.logprobs.content:
                    logprobs_buffer.extend(choice.logprobs.content)

                content_text = assistant_message.content or ''
                
                answer = self._scan_for_answer(content_text)
                if answer is not None:
                    final_answer = answer
                    break

                tool_calls = assistant_message.tool_calls
                code = self._extract_tool_code(tool_calls)

                if code is not None:
                    python_calls += 1
                    
                    messages.append({
                        'role': 'assistant',
                        'content': content_text if content_text else None,
                        'tool_calls': [
                            {
                                'id': tc.id,
                                'type': 'function',
                                'function': {
                                    'name': tc.function.name,
                                    'arguments': tc.function.arguments
                                }
                            }
                            for tc in tool_calls
                        ]
                    })

                    output = local_tool.execute(code)

                    if output.startswith('[ERROR]') or 'Traceback' in output or 'Error:' in output:
                        python_errors += 1

                    messages.append({
                        'role': 'tool',
                        'tool_call_id': tool_calls[0].id,
                        'content': output
                    })

                else:
                    if choice.finish_reason == 'stop':
                        final_answer = self._scan_for_answer(content_text)
                    break
    
        except Exception as exc:
            python_errors += 1
    
        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)
    
        mean_entropy = self._compute_mean_entropy(logprobs_buffer)
    
        return {
            'Attempt': attempt_index + 1, 
            'Response Length': total_tokens, 
            'Python Calls': python_calls, 
            'Python Errors': python_errors, 
            'Entropy': mean_entropy, 
            'Answer': final_answer
        }
    
    def _select_answer(self, detailed_results: list) -> int:

        answer_weights = defaultdict(float)
        answer_votes = defaultdict(int)

        for result in detailed_results:
            answer = result['Answer']
            entropy = result['Entropy']
            
            if answer is not None:
                weight = 1.0 / max(entropy, 1e-9)
                
                answer_weights[answer] += weight
                answer_votes[answer] += 1

        scored_answers = []

        for answer, total_weight in answer_weights.items():
            scored_answers.append({
                'answer': answer, 
                'votes': answer_votes[answer], 
                'score': total_weight
            })

        scored_answers.sort(key=lambda x: x['score'], reverse=True)

        vote_data = []

        for item in scored_answers:
            vote_data.append((
                item['answer'], 
                item['votes'], 
                item['score']
            ))

        vote_dataframe = pd.DataFrame(
            vote_data, 
            columns=['Answer', 'Votes', 'Score']
        )

        vote_dataframe = vote_dataframe.round({'Score': 3})
        display(vote_dataframe)
        
        if not scored_answers:
            print('\nFinal Answer: 0\n')
            return 0

        final_answer = scored_answers[0]['answer']    
        print(f'\nFinal Answer: {final_answer}\n')

        return final_answer
    
    def solve_problem(self, problem: str) -> int:
    
        print(f'\nProblem: {problem}\n')
        
        user_input = f'{problem}\n\n{self.cfg.preference_prompt}'
    
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout
    
        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)
    
        deadline = time.time() + budget
    
        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')
    
        tasks = []
    
        for attempt_index in range(self.cfg.attempts):
            tasks.append((self.cfg.system_prompt, attempt_index))
    
        detailed_results = []
        valid_answers = []
    
        stop_event = threading.Event()
    
        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)
    
        try:
            futures = []
    
            for (system_prompt, attempt_index) in tasks:
                future = executor.submit(
                    self._process_attempt, 
                    user_input, 
                    system_prompt, 
                    attempt_index, 
                    stop_event, 
                    deadline
                )
    
                futures.append(future)
    
            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)
    
                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])
    
                    counts = Counter(valid_answers).most_common(1)
    
                    if counts and counts[0][1] >= self.cfg.early_stop:
                        stop_event.set()
    
                        for f in futures:
                            f.cancel()
    
                        break
    
                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue
    
        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)
            
            self.problems_remaining = max(0, self.problems_remaining - 1)
    
        if detailed_results:
            results_dataframe = pd.DataFrame(detailed_results)
            results_dataframe['Entropy'] = results_dataframe['Entropy'].round(3)
            results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64')
            
            display(results_dataframe)
    
        if not valid_answers:
            print('\nResult: 0\n')
    
            return 0
    
        return self._select_answer(detailed_results)
    
    def __del__(self):
    
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
    
        if hasattr(self, 'log_file'):
            self.log_file.close()
    
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
    
                except Exception:
                    pass

In [14]:
solver = AIMO3Solver(CFG)

Loading model weights from /kaggle/input/models/qwen-lm/qwen-3-5/transformers/qwen3.5-27b/1 into OS Page Cache...
Processed 67 files (111.17 GB) in 128.03 seconds.

Waiting for vLLM server...
Server is ready (took 90.10 seconds).

Initializing 16 persistent Jupyter kernels...
Kernels initialized in 2.80 seconds.



In [15]:
# def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    
#     id_value = id_.item(0)
#     question_text = question.item(0)
    
#     gc.disable()
    
#     final_answer = solver.solve_problem(question_text)
    
#     gc.enable()
#     gc.collect()
    
#     return pl.DataFrame({'id': id_value, 'answer': final_answer})

In [16]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    global correct_count, total_count, predictions
    
    question_id = id_.item(0)
    question_text = question.item(0)
    
    print('------')
    print(f'ID: {question_id}')
    print(f'Question: {question_text[:200]}...')
    
    final_answer = solver.solve_problem(question_text)
    predictions[question_id] = final_answer

    total_count += 1
    if question_id in ground_truth:
        gt = ground_truth[question_id]
        is_correct = (final_answer == gt)
        if is_correct:
            correct_count += 1
        status = 'CORRECT' if is_correct else 'WRONG'
        print(f'Answer: {final_answer} | Ground Truth: {gt} | {status}')
        print(f'Running Accuracy: {correct_count}/{total_count} ({100*correct_count/total_count:.1f}%)')
    else:
        print(f'Answer: {final_answer}')
    
    print('------\n')
    
    return pl.DataFrame({'id': question_id, 'answer': final_answer})

In [17]:
# ── Set to True for local validation with external dataset, False for competition ──
USE_LOCAL_DATASET = True
LOCAL_DATASET_PATH = '/kaggle/input/datasets/nahidhossainredom/omni-math-hardestdifficulty-9/omni_math_hard.csv'

ground_truth = {}
predictions = {}
correct_count = 0
total_count = 0

if USE_LOCAL_DATASET:
    df = pd.read_csv(LOCAL_DATASET_PATH)
    df = df[['id', 'problem', 'answer']]
    ground_truth = dict(zip(df['id'], df['answer']))
    df.drop('answer', axis=1, errors='ignore').to_csv('reference.csv', index=False)
    print(f'Local validation dataset loaded: {len(df)} problems.')

Local validation dataset loaded: 28 problems.


In [18]:
inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
    
else:
    if USE_LOCAL_DATASET:
        inference_server.run_local_gateway(('reference.csv',))
    else:
        inference_server.run_local_gateway(
            ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',)
        )

------
ID: 1779
Question: Let $\mathcal{A}$ denote the set of all polynomials in three variables $x, y, z$ with integer coefficients. Let $\mathcal{B}$ denote the subset of $\mathcal{A}$ formed by all polynomials which can be ...

Problem: Let $\mathcal{A}$ denote the set of all polynomials in three variables $x, y, z$ with integer coefficients. Let $\mathcal{B}$ denote the subset of $\mathcal{A}$ formed by all polynomials which can be expressed as
\begin{align*}
(x + y + z)P(x, y, z) + (xy + yz + zx)Q(x, y, z) + xyzR(x, y, z)
\end{align*}
with $P, Q, R \in \mathcal{A}$.  Find the smallest non-negative integer $n$ such that $x^i y^j z^k \in \mathcal{B}$ for all non-negative integers $i, j, k$ satisfying $i + j + k \geq n$.

Budget: 900.00 seconds | Deadline: 1772215599.39



,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,6,0,0,0,inf,<NA>
1,7,0,0,0,inf,<NA>
2,3,0,0,0,inf,<NA>
3,8,0,0,0,inf,<NA>
4,4,0,0,0,inf,<NA>
5,5,0,0,0,inf,<NA>
6,1,0,0,0,inf,<NA>
7,2,0,0,0,inf,<NA>



Result: 0

Answer: 0 | Ground Truth: 4 | WRONG
Running Accuracy: 0/1 (0.0%)
------

------
ID: 1755
Question: A [i]site[/i] is any point $(x, y)$ in the plane such that $x$ and $y$ are both positive integers less than or equal to 20.

Initially, each of the 400 sites is unoccupied. Amy and Ben take turns plac...

Problem: A [i]site[/i] is any point $(x, y)$ in the plane such that $x$ and $y$ are both positive integers less than or equal to 20.

Initially, each of the 400 sites is unoccupied. Amy and Ben take turns placing stones with Amy going first. On her turn, Amy places a new red stone on an unoccupied site such that the distance between any two sites occupied by red stones is not equal to $\sqrt{5}$. On his turn, Ben places a new blue stone on any unoccupied site. (A site occupied by a blue stone is allowed to be at any distance from any other occupied site.) They stop as soon as a player cannot place a stone.

Find the greatest $K$ such that Amy can ensure that she places at lea

,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,1,0,0,0,inf,<NA>
1,2,0,0,0,inf,<NA>
2,4,0,0,0,inf,<NA>
3,3,0,0,0,inf,<NA>
4,5,0,0,0,inf,<NA>
5,6,0,0,0,inf,<NA>
6,8,0,0,0,inf,<NA>
7,7,0,0,0,inf,<NA>



Result: 0

Answer: 0 | Ground Truth: 100 | WRONG
Running Accuracy: 0/2 (0.0%)
------

------
ID: 1740
Question: Turbo the snail plays a game on a board with $2024$ rows and $2023$ columns. There are hidden monsters in $2022$ of the cells. Initially, Turbo does not know where any of the monsters are, but he know...

Problem: Turbo the snail plays a game on a board with $2024$ rows and $2023$ columns. There are hidden monsters in $2022$ of the cells. Initially, Turbo does not know where any of the monsters are, but he knows that there is exactly one monster in each row except the first row and the last row, and that each column contains at most one monster.

Turbo makes a series of attempts to go from the first row to the last row. On each attempt, he chooses to start on any cell in the first row, then repeatedly moves to an adjacent cell sharing a common side. (He is allowed to return to a previously visited cell.) If he reaches a cell with a monster, his attempt ends and he is transpo

...


Result: 0

Answer: 0 | Ground Truth: 2 | WRONG
Running Accuracy: 0/26 (0.0%)
------

------
ID: 34
Question: FIx positive integer $n$. Prove: For any positive integers $a,b,c$ not exceeding $3n^2+4n$, there exist integers $x,y,z$ with absolute value not exceeding $2n$ and not all $0$, such that $ax+by+cz=0$...

Problem: FIx positive integer $n$. Prove: For any positive integers $a,b,c$ not exceeding $3n^2+4n$, there exist integers $x,y,z$ with absolute value not exceeding $2n$ and not all $0$, such that $ax+by+cz=0$

Budget: 900.00 seconds | Deadline: 1772215603.63



,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,2,0,0,0,inf,<NA>
1,3,0,0,0,inf,<NA>
2,1,0,0,0,inf,<NA>
3,4,0,0,0,inf,<NA>
4,5,0,0,0,inf,<NA>
5,6,0,0,0,inf,<NA>
6,7,0,0,0,inf,<NA>
7,8,0,0,0,inf,<NA>



Result: 0

Answer: 0 | Ground Truth: 0 | CORRECT
Running Accuracy: 1/27 (3.7%)
------

------
ID: 1805
Question: 2500 chess kings have to be placed on a $100 \times 100$ chessboard so that

[b](i)[/b] no king can capture any other one (i.e. no two kings are placed in two squares sharing a common vertex);
[b](ii)...

Problem: 2500 chess kings have to be placed on a $100 \times 100$ chessboard so that

[b](i)[/b] no king can capture any other one (i.e. no two kings are placed in two squares sharing a common vertex);
[b](ii)[/b] each row and each column contains exactly 25 kings.

Find the number of such arrangements. (Two arrangements differing by rotation or symmetry are supposed to be different.)

[i]

Budget: 900.00 seconds | Deadline: 1772215603.72



,Attempt,Response Length,Python Calls,Python Errors,Entropy,Answer
0,2,0,0,0,inf,<NA>
1,1,0,0,0,inf,<NA>
2,5,0,0,0,inf,<NA>
3,4,0,0,0,inf,<NA>
4,3,0,0,0,inf,<NA>
5,7,0,0,0,inf,<NA>
6,8,0,0,0,inf,<NA>
7,6,0,0,0,inf,<NA>



Result: 0

Answer: 0 | Ground Truth: 2 | WRONG
Running Accuracy: 1/28 (3.6%)
------

